<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/20-efficient-inference-deployment.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **高效推理与部署** {#efficient-inference-deployment}

推理是把训练完成的模型转化为满足运行约束的决策过程。Notebook 只需要回答模型能否产生正确张量；部署系统还必须验证请求、满足延迟目标、在多个用户之间共享有限内存、处理异常输入、暴露遥测指标，并支持回滚。因此，“更快的推理”并不是一项孤立优化，而是围绕**模型质量、请求调度、数值表示、运行时编译、硬件与运维**展开的联合设计问题。

本章延续第 19 章的 UCI Optical Recognition of Handwritten Digits 实验主线。[UCI 数据集](https://doi.org/10.24432/C50P49)包含 1,797 张带标签的 8 x 8 图像，采用 **CC BY 4.0** 许可。分类部署实验共享一个确定性的训练/验证/测试划分。对于自回归机制，每张图像被转换为由起始 token、类别 token、64 个整数像素强度 token 和结束 token 组成的序列。这个派生序列仍然对应同一批观测数据；它只用于验证缓存与精确 speculative sampling 的机制，并不是语言模型 benchmark。

<details>
<summary><strong>PyTorch：建立整章共享的 Digits 部署工作负载</strong></summary>

```python
import copy
import math
import random
import time

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=2020):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_ids = np.arange(len(digits.data))
train_ids, remaining_ids = train_test_split(
    all_ids, test_size=0.30, stratify=digits.target, random_state=2020
)
val_ids, test_ids = train_test_split(
    remaining_ids,
    test_size=0.50,
    stratify=digits.target[remaining_ids],
    random_state=2020,
)

# UCI documents pixel intensities on 0..16, so this fixed divisor does not leak test statistics.
features = torch.tensor(digits.data / 16.0, dtype=torch.float32)
targets = torch.tensor(digits.target, dtype=torch.long)
train_dataset = TensorDataset(features[train_ids], targets[train_ids])
val_dataset = TensorDataset(features[val_ids], targets[val_ids])
test_dataset = TensorDataset(features[test_ids], targets[test_ids])
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(2020),
)
val_loader = DataLoader(val_dataset, batch_size=128)
test_loader = DataLoader(test_dataset, batch_size=128)


class DigitMLP(nn.Module):
    def __init__(self, hidden1=128, hidden2=64):
        super().__init__()
        self.fc1 = nn.Linear(64, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, 10)

    def forward(self, x):
        x = F.gelu(self.fc1(x))
        x = F.gelu(self.fc2(x))
        return self.fc3(x)


def accuracy(model, loader):
    model.eval()
    correct = total = 0
    with torch.inference_mode():
        for x, y in loader:
            correct += int((model(x).argmax(1) == y).sum())
            total += len(y)
    return correct / total


seed_everything()
baseline_model = DigitMLP()
optimizer = torch.optim.AdamW(baseline_model.parameters(), lr=2e-3, weight_decay=1e-4)
for _ in range(15):
    baseline_model.train()
    for x, y in train_loader:
        loss = F.cross_entropy(baseline_model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# Reuse the same observations as short discrete sequences for generative mechanisms.
BOS, LABEL_OFFSET, EOS, VOCAB_SIZE = 27, 17, 28, 29
pixel_tokens = torch.tensor(digits.data.astype(np.int64), dtype=torch.long)
token_sequences = torch.cat(
    [
        torch.full((len(digits.data), 1), BOS, dtype=torch.long),
        targets[:, None] + LABEL_OFFSET,
        pixel_tokens,
        torch.full((len(digits.data), 1), EOS, dtype=torch.long),
    ],
    dim=1,
)

baseline_accuracy = accuracy(baseline_model, test_loader)
assert len(set(train_ids) & set(test_ids)) == 0
assert token_sequences.shape == (1797, 67) and baseline_accuracy > 0.92
print({"split": (len(train_ids), len(val_ids), len(test_ids)), "test_accuracy": round(baseline_accuracy, 3)})
```

</details>

基线模型足够小，可以直接在 CPU 上运行。除非正文明确指出生产运行时，下面的计时都只描述当前本地 eager 环境；它们用于演示测量与决策流程，而不是给出跨硬件都成立的性能排名。


### **推理生命周期** {#inference-lifecycle}

一次前向传播把合法张量映射为 logits；推理生命周期开始得更早、结束得更晚。请求首先被接纳、鉴权和解析，预处理必须复现训练时的数据契约，调度器把请求组织成可执行工作，运行时选择 kernel 与设备内存，后处理把张量转换为应用响应，遥测系统则记录全过程。即使模型本身正确，只要 tokenizer 不一致、归一化规则过期或请求形状没有边界，服务仍然是错误的。

![从请求验证到运行观测的推理生命周期。](assets/dl20-inference-lifecycle.svg){fig-align="center" width="76%" fig-alt="五阶段流程依次经过请求验证、准备、模型执行、后处理与观测，下方是发布契约。"}

*原创教学图，依据 [ONNX Runtime](https://onnxruntime.ai/docs/) 描述的部署边界以及 [NVIDIA Triton Inference Server](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/) 的请求服务职责整理。*

一份有用的接口契约至少应规定输入 schema、允许的形状与 dtype、预处理版本、输出语义、模型产物摘要、超时时间与失败行为。幂等的 request ID 让重试过程可以被追踪。readiness check 应加载真实产物并运行代表性输入；liveness check 只应确认进程仍能响应，否则一次暂时性的加速器故障可能触发重启风暴。

<details>
<summary><strong>PyTorch：用明确的请求-响应契约封装模型</strong></summary>

```python
MODEL_VERSION = "digits-mlp-2020-v1"


def predict_digits(request):
    # 1. Validate the external schema before allocating an unbounded tensor.
    request_id = str(request["request_id"])
    raw = np.asarray(request["pixels"], dtype=np.float32)
    if raw.shape not in {(8, 8), (64,)}:
        raise ValueError("pixels must have shape (8, 8) or (64,)")
    if not np.isfinite(raw).all() or raw.min() < 0 or raw.max() > 16:
        raise ValueError("pixels must be finite values in the documented 0..16 range")

    # 2. Apply the same fixed preprocessing contract used during training.
    model_input = torch.from_numpy(raw.reshape(1, 64) / 16.0)

    # 3. Disable autograd and execute the immutable evaluation model.
    started_ns = time.perf_counter_ns()
    baseline_model.eval()
    with torch.inference_mode():
        probabilities = baseline_model(model_input).softmax(dim=-1)[0]
    elapsed_ms = (time.perf_counter_ns() - started_ns) / 1e6

    # 4. Return stable application semantics and enough identity for tracing.
    top_probability, top_class = probabilities.max(dim=0)
    return {
        "request_id": request_id,
        "model_version": MODEL_VERSION,
        "prediction": int(top_class),
        "confidence": float(top_probability),
        "model_ms": elapsed_ms,
    }


sample_id = int(test_ids[0])
response = predict_digits({"request_id": "demo-1", "pixels": digits.images[sample_id]})
assert response["prediction"] in range(10) and 0.0 <= response["confidence"] <= 1.0
try:
    predict_digits({"request_id": "bad", "pixels": np.zeros((16, 16))})
    raise AssertionError("shape validation should have failed")
except ValueError:
    pass
print(response)
```

</details>

这个封装把**模型延迟**与端到端延迟分开。该区别非常重要：即使模型 kernel 正常，排队或图像解码仍然可能让响应变慢。每次切换候选产物之前，都应使用同一份契约进行重放测试。


### **延迟、吞吐量与尾部行为** {#latency-throughput-tail}

延迟是单个请求经历的时间，吞吐量是单位时间内完成的工作量。对请求 (i)，有

$$L_i = t_i^{\text{return}} - t_i^{\text{arrival}}, \qquad \lambda = \frac{N_{\text{completed}}}{\Delta t}.$$

服务级目标通常是 (P(L \leq L_{\mathrm{SLO}}) \geq 0.99) 这样的分位数约束，而不是平均值。第 99 百分位能够暴露均值掩盖的慢尾部。排队会把两个指标耦合起来：当输入负载接近服务容量时，很小的突发流量也可能造成不成比例的等待。对稳定系统，Little 定律 (Q=\lambda W) 连接平均在系统中的请求数 (Q)、实际吞吐量 (\lambda) 与平均停留时间 (W)。

![标出中位数与尾部延迟的累积分布曲线。](assets/dl20-latency-tail.svg){fig-align="center" width="72%" fig-alt="累积延迟曲线标出 p50 与 p99，并把排队、形状回退和缓存未命中列为尾部原因。"}

对自回归生成而言，一个延迟数字还不够。**首 token 时间（TTFT）**包括排队和 prompt prefill，**token 间延迟（ITL）**或**每个输出 token 时间（TPOT）**描述 decode 节奏，端到端延迟还取决于输出长度。交互式系统通常优先关注 TTFT 与尾部 ITL，离线任务则更看重每秒 token 数。

<details>
<summary><strong>Python：测量预热后的同步延迟分布与吞吐量</strong></summary>

```python
def benchmark_classifier(model, requests, batch_size=1, repeats=3):
    model.eval()
    # Warm-up removes one-time initialization from the steady-state sample.
    with torch.inference_mode():
        for _ in range(20):
            model(requests[: min(batch_size, len(requests))])

    samples_ms = []
    started = time.perf_counter()
    completed = 0
    with torch.inference_mode():
        for _ in range(repeats):
            for start in range(0, len(requests), batch_size):
                batch = requests[start : start + batch_size]
                tick = time.perf_counter_ns()
                model(batch)
                samples_ms.append((time.perf_counter_ns() - tick) / 1e6)
                completed += len(batch)
    elapsed = time.perf_counter() - started
    return {
        "p50_batch_ms": float(np.percentile(samples_ms, 50)),
        "p95_batch_ms": float(np.percentile(samples_ms, 95)),
        "p99_batch_ms": float(np.percentile(samples_ms, 99)),
        "requests_per_second": completed / elapsed,
    }


test_x = features[test_ids]
single_metrics = benchmark_classifier(baseline_model, test_x[:128], batch_size=1)
batch_metrics = benchmark_classifier(baseline_model, test_x[:128], batch_size=32)
assert single_metrics["p99_batch_ms"] >= single_metrics["p50_batch_ms"]
assert batch_metrics["requests_per_second"] > 0
print({"single": single_metrics, "batch_32": batch_metrics})
```

</details>

计时报告必须记录硬件、运行时版本、线程数、batch/sequence 形状、精度、预热方式、并发度以及是否同步设备执行。比较时应使用接近真实情况的到达过程；紧凑的 microbenchmark 可以暴露 kernel 成本，却无法预测网络排队或多租户干扰。


### **静态、动态与连续批处理** {#static-dynamic-continuous-batching}

批处理可以摊薄调度开销并暴露并行工作，但也可能让较早到达的请求等待其他请求。**静态批处理**在执行前固定 batch 形状，适合同质的离线数据。**动态批处理**收集相容请求，直到达到最大 batch size 或队列超时。**连续批处理**面向长度不一的自回归解码：序列完成后立即释放槽位，等待中的序列可以进入，而不必等原始 batch 的所有成员都结束。

![静态、动态和连续批处理的时间线比较。](assets/dl20-batching.svg){fig-align="center" width="76%" fig-alt="三条时间线分别展示固定完整 batch、由大小或超时形成的 batch，以及持续补充的序列槽位。"}

*原创教学图，依据 [Triton dynamic batcher](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/batcher.html) 以及现代生成式服务系统使用的 iteration-level scheduling 整理。*

相容性与到达时间同样重要。请求可能必须共享模型版本、精度、adapter、张量形状 bucket 或解码策略。对变化很大的序列做 padding 会浪费计算；把队列拆得过细又会损失利用率。因此，调度器优化的是“在 (p99 \leq L_{\mathrm{SLO}}) 条件下最大化吞吐量”一类受约束目标，而不是单独追求更大的 batch。

<details>
<summary><strong>Python：使用 Digits 派生请求模拟动态与连续调度器</strong></summary>

```python
# Deterministic inter-arrival gaps and generation lengths come from test observations.
arrival_ms = np.cumsum(0.15 + (digits.target[test_ids[:80]] % 4) * 0.08)
token_budgets = 4 + (pixel_tokens[test_ids[:80]] > 0).sum(dim=1).numpy() // 4


def dynamic_batches(arrivals, max_batch=12, timeout_ms=0.8):
    batches, queue = [], []
    first_arrival = None
    for request_id, arrival in enumerate(arrivals):
        if queue and arrival - first_arrival >= timeout_ms:
            batches.append(queue)
            queue, first_arrival = [], None
        if not queue:
            first_arrival = arrival
        queue.append(request_id)
        if len(queue) == max_batch:
            batches.append(queue)
            queue, first_arrival = [], None
    if queue:
        batches.append(queue)
    return batches


def continuous_decode(lengths, slots=8):
    waiting = list(enumerate(lengths))
    active, finish_step = [], {}
    step = 0
    while waiting or active:
        while waiting and len(active) < slots:
            request_id, length = waiting.pop(0)
            active.append([request_id, int(length)])
        step += 1
        for item in active:
            item[1] -= 1
        finished = [item for item in active if item[1] == 0]
        for request_id, _ in finished:
            finish_step[request_id] = step
        active = [item for item in active if item[1] > 0]
    return finish_step, step


formed = dynamic_batches(arrival_ms)
continuous_finish, continuous_steps = continuous_decode(token_budgets, slots=8)
padded_steps = sum(max(token_budgets[i : i + 8]) for i in range(0, len(token_budgets), 8))
assert sum(map(len, formed)) == len(arrival_ms)
assert set(continuous_finish) == set(range(len(token_budgets)))
print({
    "dynamic_batch_sizes": [len(batch) for batch in formed[:6]],
    "continuous_steps": continuous_steps,
    "fixed_padded_steps": int(padded_steps),
})
```

</details>

连续批处理并不会让每个请求都更快。它提高的是槽位利用率，而 admission control 与抢占策略决定谁需要等待。应分别报告排队延迟和执行时间，并按照 prompt 长度、输出长度、adapter 与优先级检查延迟，避免良好的总指标掩盖饥饿问题。


### **KV Cache 与内存管理** {#kv-cache-memory-management}

在因果自注意力中，第 (t) 个解码位置需要位置 (1,\ldots,t) 的 key 与 value。如果每生成一个 token 都重新计算全部旧投影，就会重复大量工作。**KV cache** 在 prefill 后保存已有的 key/value 张量，并在每个 decode step 追加一个位置。新的 query 仍然要关注整个前缀，但旧的 key 与 value 可以直接复用。

对 (N_L) 层、batch size (B)、缓存长度 (T)、(H_{kv}) 个 key/value head、head dimension (d_h) 以及每个元素 (b) 字节，缓存的主要内存为

$$M_{KV} = 2N_LBTH_{kv}d_hb.$$

系数 2 代表 key 与 value。Multi-query attention 只使用一个 KV head，grouped-query attention 使用少于 query head 数量的 KV head，因此都能降低这项内存。静态 cache 预留已知最大形状，便于编译但可能浪费容量；动态 cache 自然增长，却会增加分配复杂度。[Hugging Face cache 指南](https://huggingface.co/docs/transformers/main/kv_cache)记录了这种速度-内存取舍。

![KV cache 的前缀复用与分页式内存管理。](assets/dl20-kv-cache.svg){fig-align="center" width="76%" fig-alt="Prefill 阶段把 key/value block 写入分页 cache，每个 decode step 读取前缀并追加一个 block。"}

当每个请求都获得一块很大的连续预留空间时，不同序列长度会造成碎片。[PagedAttention](https://arxiv.org/abs/2309.06180)像操作系统虚拟内存分页一样，把逻辑 KV block 映射到不连续的物理 block，从而改善分配与前缀共享；但它不会消除 cache 内容随上下文长度线性增长的事实。

<details>
<summary><strong>PyTorch：验证完整因果注意力与增量 KV cache 的等价性</strong></summary>

```python
seed_everything(2020)
sequence = token_sequences[int(test_ids[0]), :18]
model_dim, heads, head_dim = 32, 4, 8
embedding = nn.Embedding(VOCAB_SIZE, model_dim)
q_proj = nn.Linear(model_dim, model_dim, bias=False)
k_proj = nn.Linear(model_dim, model_dim, bias=False)
v_proj = nn.Linear(model_dim, model_dim, bias=False)


def split_heads(x):
    batch, length, _ = x.shape
    return x.view(batch, length, heads, head_dim).transpose(1, 2)


hidden = embedding(sequence[None, :])
q_full, k_full, v_full = map(split_heads, (q_proj(hidden), k_proj(hidden), v_proj(hidden)))
scores = q_full @ k_full.transpose(-1, -2) / math.sqrt(head_dim)
causal_mask = torch.triu(torch.ones(len(sequence), len(sequence), dtype=torch.bool), diagonal=1)
full_output = torch.softmax(scores.masked_fill(causal_mask, -torch.inf), dim=-1) @ v_full

cached_k = cached_v = None
incremental_outputs = []
for position in range(len(sequence)):
    token_hidden = embedding(sequence[position : position + 1][None, :])
    q_t = split_heads(q_proj(token_hidden))
    k_t = split_heads(k_proj(token_hidden))
    v_t = split_heads(v_proj(token_hidden))
    cached_k = k_t if cached_k is None else torch.cat([cached_k, k_t], dim=2)
    cached_v = v_t if cached_v is None else torch.cat([cached_v, v_t], dim=2)
    attention_t = torch.softmax(q_t @ cached_k.transpose(-1, -2) / math.sqrt(head_dim), dim=-1)
    incremental_outputs.append(attention_t @ cached_v)

incremental_output = torch.cat(incremental_outputs, dim=2)


def kv_cache_mib(layers, batch, tokens, kv_heads, dimension, bytes_per_value):
    return 2 * layers * batch * tokens * kv_heads * dimension * bytes_per_value / 2**20


assert torch.allclose(full_output, incremental_output, atol=1e-6)
print({
    "equivalent": True,
    "example_cache_MiB": round(kv_cache_mib(32, 8, 4096, 8, 128, 2), 1),
})
```

</details>

cache 正确性测试必须覆盖位置索引、attention mask、beam 重排、前缀所有权与驱逐。运行诊断则应记录已分配/已使用 block、cache hit rate、碎片率、驱逐率以及每个 active token 的字节数。即使 cache 策略提高了 batch 容量，只要 offloading 或驱逐导致重复传输，延迟仍然可能恶化。


### **Speculative Decoding** {#speculative-decoding}

自回归解码经常受内存带宽限制，因为大型 target model 每次调用只生成一个新 token。**Speculative decoding** 让较便宜的 draft model (q) 先提出多个 token，再由 target model (p) 并行验证。核心要求是分布精确性：被拒绝的 draft token 必须经过校正，使最终输出仍服从 (p)，而不是仅为速度选择的近似分布。

对一个提议 token (y\sim q)，以如下概率接受：

$$\alpha(y)=\min\left(1,\frac{p(y)}{q(y)}\right).$$

如果被拒绝，则从归一化残差 (r(y)\propto\max(0,p(y)-q(y))) 中采样。这个接受-拒绝恒等式保证边缘输出恰好服从 (p)。[原始 speculative decoding 论文](https://proceedings.mlr.press/v202/leviathan23a.html)把该思想扩展到 token block，用更少的串行 target 调用完成验证。

![Speculative decoding 中的 draft 提议、target 验证与精确校正。](assets/dl20-speculative.svg){fig-align="center" width="74%" fig-alt="Draft model 提出 token block，target model 用接受概率验证，并用已接受 token 或校正样本扩展提交前缀。"}

<details>
<summary><strong>Python：在 Digits token 转移上验证精确的一步 speculative sampling</strong></summary>

```python
# Build a target bigram distribution and a cheaper smoothed draft from training sequences only.
transition_counts = np.full((VOCAB_SIZE, VOCAB_SIZE), 0.5, dtype=np.float64)
unigram_counts = np.full(VOCAB_SIZE, 0.5, dtype=np.float64)
for sequence_row in token_sequences[train_ids].numpy():
    np.add.at(transition_counts, (sequence_row[:-1], sequence_row[1:]), 1)
    np.add.at(unigram_counts, sequence_row[1:], 1)
target_bigram = transition_counts / transition_counts.sum(axis=1, keepdims=True)
unigram = unigram_counts / unigram_counts.sum()
draft_bigram = 0.82 * target_bigram + 0.18 * unigram[None, :]


def speculative_one_step(p, q, rng):
    proposal = int(rng.choice(len(q), p=q))
    acceptance = min(1.0, p[proposal] / q[proposal])
    if rng.random() <= acceptance:
        return proposal, True
    residual = np.maximum(p - q, 0.0)
    residual /= residual.sum()
    return int(rng.choice(len(p), p=residual)), False


context = int(token_sequences[int(test_ids[0]), 8])
p, q = target_bigram[context], draft_bigram[context]
rng = np.random.default_rng(2020)
draws, accepted = zip(*(speculative_one_step(p, q, rng) for _ in range(20000)))
empirical = np.bincount(draws, minlength=VOCAB_SIZE) / len(draws)
total_variation = 0.5 * np.abs(empirical - p).sum()
assert total_variation < 0.035
print({"acceptance_rate": round(np.mean(accepted), 3), "target_TV_error": round(total_variation, 4)})
```

</details>

加速效果取决于接受长度、draft 成本、target 验证效率、采样设置与同步开销。draft 太弱会频繁拒绝，太大则节省不了多少工作。应在真实解码策略下测量每次 target 调用接受的 token 数以及端到端 TTFT/TPOT。Greedy assisted decoding 更简单，但不能与上述精确采样算法混为一谈。


### **训练后量化** {#post-training-quantization}

量化使用更小的离散代码表示实数。仿射映射由 scale (s>0)、zero point (z) 与整数范围 ([q_{\min},q_{\max}]) 构成：

$$q=\operatorname{clip}\left(\operatorname{round}(x/s)+z,q_{\min},q_{\max}\right), \qquad \hat{x}=s(q-z).$$

(q) 由整数 kernel 存储或处理，(\hat{x}) 是重构值。对称有符号量化设 (z=0)，对 (b) bit 通常取 (s=\max|x|/(2^{b-1}-1))。逐通道 weight scale 能适应幅值不同的行；逐张量 activation scale 管理成本更低。截断虽然缩小了范围，却可能提高主要分布区域的分辨率。

**训练后量化（PTQ）**在不重新优化任务 loss 的情况下修改训练完成的模型。动态 PTQ 在执行时估计 activation 参数；静态 PTQ 预先运行具有代表性的**训练/校准**数据，并把 activation 参数写入产物。[ONNX Runtime 量化指南](https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html)区分了这两种方式，并建议在质量回退时比较对应的 FP32 与量化 activation。

![训练后校准、转换和验证流程。](assets/dl20-ptq.svg){fig-align="center" width="76%" fig-alt="流程从浮点模型经过校准和量化进入验证，并包含 activation 级别的调试路径。"}

<details>
<summary><strong>PyTorch：实现经过校准的 W8A8 训练后量化</strong></summary>

```python
def symmetric_scale(x, bits=8, dim=None):
    limit = 2 ** (bits - 1) - 1
    maximum = x.detach().abs().amax(dim=dim, keepdim=dim is not None)
    return maximum.clamp_min(1e-8) / limit


def quantize_dequantize(x, scale, bits=8):
    limit = 2 ** (bits - 1) - 1
    return torch.round(x / scale).clamp(-limit, limit) * scale


# Observe post-activation ranges using training data only.
activation_max = {"input": 0.0, "hidden1": 0.0, "hidden2": 0.0}
baseline_model.eval()
with torch.inference_mode():
    for x, _ in train_loader:
        h1 = F.gelu(baseline_model.fc1(x))
        h2 = F.gelu(baseline_model.fc2(h1))
        activation_max["input"] = max(activation_max["input"], float(x.abs().max()))
        activation_max["hidden1"] = max(activation_max["hidden1"], float(h1.abs().max()))
        activation_max["hidden2"] = max(activation_max["hidden2"], float(h2.abs().max()))
activation_scales = {name: torch.tensor(value / 127.0) for name, value in activation_max.items()}


class StaticPTQDigitMLP(nn.Module):
    def __init__(self, source, scales):
        super().__init__()
        self.fc1, self.fc2, self.fc3 = [copy.deepcopy(layer) for layer in (source.fc1, source.fc2, source.fc3)]
        self.scales = scales
        with torch.no_grad():
            for layer in (self.fc1, self.fc2, self.fc3):
                row_scale = symmetric_scale(layer.weight, bits=8, dim=1)
                layer.weight.copy_(quantize_dequantize(layer.weight, row_scale, bits=8))

    def forward(self, x):
        x = quantize_dequantize(x, self.scales["input"], bits=8)
        x = F.gelu(self.fc1(x))
        x = quantize_dequantize(x, self.scales["hidden1"], bits=8)
        x = F.gelu(self.fc2(x))
        x = quantize_dequantize(x, self.scales["hidden2"], bits=8)
        return self.fc3(x)


ptq_model = StaticPTQDigitMLP(baseline_model, activation_scales)
with torch.inference_mode():
    reference_logits = baseline_model(test_x[:128])
    ptq_logits = ptq_model(test_x[:128])
ptq_accuracy = accuracy(ptq_model, test_loader)
ptq_logit_mae = float((reference_logits - ptq_logits).abs().mean())
assert ptq_accuracy > 0.90 and ptq_logit_mae > 0
print({"FP32": round(baseline_accuracy, 3), "PTQ_W8A8": round(ptq_accuracy, 3), "logit_MAE": round(ptq_logit_mae, 4)})
```

</details>

为了不依赖特定整数后端，示例模型仍以重构后的张量保存参数；它验证的是量化数值，而不是 INT8 速度。真实加速还需要受支持的打包算子与硬件指令。校准数据必须覆盖部署范围，同时与测试集隔离；范围过窄会使偏移输入饱和，离群值又可能浪费大部分整数级别。


### **量化感知训练** {#quantization-aware-training}

当微小舍入误差在敏感层中不断累积时，PTQ 可能失败。**量化感知训练（QAT）**在部署前让模型接触这些误差。前向传播插入 fake quantize/dequantize 操作，而可训练参数仍保持浮点。由于 rounding 几乎处处导数为零，常用 straight-through estimator（STE）用近似恒等梯度穿过量化器。

![QAT 的 fake-quantized 前向传播与 straight-through 反向循环。](assets/dl20-qat.svg){fig-align="center" width="74%" fig-alt="循环展示前向传播中的 fake-quantized weight/activation、任务 loss、straight-through gradient 与浮点参数更新。"}

QAT 并不保证可部署的加速。fake-quant graph 最终必须被转换为目标后端支持的算子，而且 observer、粒度和数值假设必须与训练一致。当前 PyTorch 的量化开发集中在 [torchao](https://docs.pytorch.org/ao/stable/workflows/qat.html)；具体 API 会演进，但 prepare-train-convert 生命周期是稳定的概念边界。

<details>
<summary><strong>PyTorch：使用 straight-through fake quantization 微调 Digits 模型</strong></summary>

```python
def ste_quantize_dequantize(x, scale, bits=8):
    reconstructed = quantize_dequantize(x, scale, bits)
    return x + (reconstructed - x).detach()


class QATDigitMLP(DigitMLP):
    def __init__(self, source, scales):
        super().__init__()
        self.load_state_dict(copy.deepcopy(source.state_dict()))
        self.scales = scales

    def quantized_linear(self, x, layer):
        weight_scale = symmetric_scale(layer.weight, bits=8, dim=1)
        weight = ste_quantize_dequantize(layer.weight, weight_scale, bits=8)
        return F.linear(x, weight, layer.bias)

    def forward(self, x):
        x = ste_quantize_dequantize(x, self.scales["input"], bits=8)
        x = F.gelu(self.quantized_linear(x, self.fc1))
        x = ste_quantize_dequantize(x, self.scales["hidden1"], bits=8)
        x = F.gelu(self.quantized_linear(x, self.fc2))
        x = ste_quantize_dequantize(x, self.scales["hidden2"], bits=8)
        return self.quantized_linear(x, self.fc3)


qat_model = QATDigitMLP(baseline_model, activation_scales)
qat_optimizer = torch.optim.AdamW(qat_model.parameters(), lr=2e-4, weight_decay=1e-4)
for _ in range(4):
    qat_model.train()
    for x, y in train_loader:
        qat_loss = F.cross_entropy(qat_model(x), y)
        qat_optimizer.zero_grad()
        qat_loss.backward()
        qat_optimizer.step()

qat_accuracy = accuracy(qat_model, test_loader)
assert qat_accuracy > 0.90 and any(parameter.grad is not None for parameter in qat_model.parameters())
print({"PTQ": round(ptq_accuracy, 3), "QAT_fake_quant": round(qat_accuracy, 3)})
```

</details>

当 PTQ 无法满足质量预算且存在代表性微调数据时，QAT 才值得付出成本。应先通过逐层 activation error、clipping frequency 与分 slice 任务指标定位问题。如果只有少数算子敏感，mixed precision 或 selective quantization 往往比全模型 QAT 更干净。


### **仅权重量化与低 bit 方法** {#weight-only-low-bit}

在大型模型中，从内存读取 weight 可能主导 decode 成本。**仅权重量化**用低精度保存 weight，同时让 activation 与累加保持较宽类型。它不需要 activation calibration，能够降低带宽，但每个矩阵乘法都需要兼容的打包 kernel 来反量化或直接处理低 bit block。

分组量化把每一行切成包含 (G) 个 weight 的 block，并让每个 block 共享一个 scale。对于 (n_w) 个 weight、(b)-bit code 与 (n_s\) 个 FP32 scale，逻辑存储量近似为

$$M \approx \frac{bn_w}{8} + 4n_s \text{ bytes}.$$

较小的 group 能更好适应局部范围，却会增加 scale 元数据。Round-to-nearest（RTN）不需要数据；GPTQ 使用校准 activation 近似最小化逐层重构误差；AWQ 利用 activation 信息保护显著 weight channel。它们代表不同的优化准则，而不是可以互换的 bit format。

![每个 block 共享一个 scale 的分组低 bit 权重存储。](assets/dl20-weight-only.svg){fig-align="center" width="76%" fig-alt="浮点矩阵被切分为彩色 group，并打包为 INT4 code 和 scale，图中同时标出粒度取舍。"}

<details>
<summary><strong>PyTorch：实现分组 INT4 权重重构与存储量核算</strong></summary>

```python
def groupwise_weight_qdq(weight, bits=4, group_size=32):
    output_dim, input_dim = weight.shape
    padding = (-input_dim) % group_size
    padded = F.pad(weight, (0, padding))
    grouped = padded.view(output_dim, -1, group_size)
    scale = symmetric_scale(grouped, bits=bits, dim=2)
    reconstructed = quantize_dequantize(grouped, scale, bits=bits)
    return reconstructed.view(output_dim, -1)[:, :input_dim], scale


int4_model = copy.deepcopy(baseline_model)
int4_scale_count = 0
with torch.no_grad():
    for layer in (int4_model.fc1, int4_model.fc2, int4_model.fc3):
        reconstructed, scales = groupwise_weight_qdq(layer.weight, bits=4, group_size=32)
        layer.weight.copy_(reconstructed)
        int4_scale_count += scales.numel()

weight_count = sum(layer.weight.numel() for layer in (baseline_model.fc1, baseline_model.fc2, baseline_model.fc3))
bias_count = sum(layer.bias.numel() for layer in (baseline_model.fc1, baseline_model.fc2, baseline_model.fc3))
fp32_bytes = 4 * (weight_count + bias_count)
int4_logical_bytes = weight_count / 2 + 4 * (int4_scale_count + bias_count)
int4_accuracy = accuracy(int4_model, test_loader)
assert int4_logical_bytes < fp32_bytes and int4_accuracy > 0.85
print({
    "FP32_KiB": round(fp32_bytes / 1024, 1),
    "INT4_logical_KiB": round(int4_logical_bytes / 1024, 1),
    "INT4_accuracy": round(int4_accuracy, 3),
})
```

</details>

代码有意重构出 FP32 weight，因此其延迟不能代表打包 INT4 kernel。当运行时插入额外转换或回退到通用算子时，低 bit 产物甚至可能更慢。选择方法之前，需要在目标硬件上验证 perplexity/任务指标、长上下文稳定性、离群层、resident memory 与每秒 token 数。


### **剪枝与知识蒸馏** {#pruning-knowledge-distillation}

**剪枝**把参数或结构设为零。非结构化 magnitude pruning 可以在保持张量形状的情况下得到高稀疏度；如果没有 sparse kernel 且元数据开销不利，它既不会减少 dense FLOPs，也不会缩短 wall time。结构化剪枝移除 channel、head、block 或 layer，使标准 dense kernel 真正看到更小的形状，但每次移除都会扰动更大的功能单元。

**知识蒸馏**训练较小 student 同时匹配 teacher 与标签。设 temperature 为 (\tau)，teacher logits 为 (z_t)，student logits 为 (z_s)，hard-label loss 为 (\mathcal{L}_{CE})，混合权重为 (\alpha)，常用目标为

$$\mathcal{L}=\alpha\mathcal{L}_{CE}(z_s,y)+(1-\alpha)\tau^2\,D_{KL}\!\left(\operatorname{softmax}(z_t/\tau)\;\|\;\operatorname{softmax}(z_s/\tau)\right).$$

soft distribution 传递了 one-hot label 没有表达的类别相似性。(\tau^2) 因子用于补偿温度缩放后的梯度幅值。

![从训练完成的 teacher 出发的剪枝与蒸馏路径。](assets/dl20-pruning-distillation.svg){fig-align="center" width="74%" fig-alt="Teacher 分支为稀疏剪枝模型和更小的蒸馏 student，两者都需要测量质量、存储和后端延迟。"}

<details>
<summary><strong>PyTorch：比较非结构化剪枝与蒸馏后的 dense student</strong></summary>

```python
pruned_model = copy.deepcopy(baseline_model)
prune_fraction = 0.50
all_magnitudes = torch.cat([layer.weight.detach().abs().flatten() for layer in (pruned_model.fc1, pruned_model.fc2, pruned_model.fc3)])
threshold = torch.quantile(all_magnitudes, prune_fraction)
zero_weights = total_weights = 0
with torch.no_grad():
    for layer in (pruned_model.fc1, pruned_model.fc2, pruned_model.fc3):
        mask = layer.weight.abs() > threshold
        layer.weight.mul_(mask)
        zero_weights += int((layer.weight == 0).sum())
        total_weights += layer.weight.numel()


class StudentMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 48)
        self.fc2 = nn.Linear(48, 10)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


seed_everything(2020)
student_model = StudentMLP()
student_optimizer = torch.optim.AdamW(student_model.parameters(), lr=2e-3, weight_decay=1e-4)
temperature, hard_weight = 2.0, 0.45
baseline_model.eval()
for _ in range(15):
    student_model.train()
    for x, y in train_loader:
        with torch.inference_mode():
            teacher_logits = baseline_model(x)
        student_logits = student_model(x)
        hard_loss = F.cross_entropy(student_logits, y)
        soft_loss = F.kl_div(
            F.log_softmax(student_logits / temperature, dim=-1),
            F.softmax(teacher_logits / temperature, dim=-1),
            reduction="batchmean",
        ) * temperature**2
        distillation_loss = hard_weight * hard_loss + (1 - hard_weight) * soft_loss
        student_optimizer.zero_grad()
        distillation_loss.backward()
        student_optimizer.step()

pruned_accuracy = accuracy(pruned_model, test_loader)
student_accuracy = accuracy(student_model, test_loader)
student_parameters = sum(parameter.numel() for parameter in student_model.parameters())
teacher_parameters = sum(parameter.numel() for parameter in baseline_model.parameters())
assert zero_weights / total_weights >= 0.49 and student_parameters < teacher_parameters
print({
    "pruned_accuracy": round(pruned_accuracy, 3),
    "sparsity": round(zero_weights / total_weights, 3),
    "student_accuracy": round(student_accuracy, 3),
    "student_parameter_ratio": round(student_parameters / teacher_parameters, 3),
})
```

</details>

剪枝保留原架构，却要求运行时利用稀疏性；蒸馏付出训练成本，换取普通 kernel 就能加速的更小 dense graph。为了证明收益来自蒸馏而不只是架构变小，应与只用 hard label 训练的 student 比较；还要检查稀有类别与校准，因为 student 也可能复制 teacher 的错误。


### **导出、编译与运行时优化** {#export-compilation-runtime-optimization}

部署会跨越三个相关但不同的边界。**导出**把程序及其输入约束捕获为可移植或 ahead-of-time 表示。**编译**针对目标变换 graph：分解算子、传播常量、融合模式、规划内存并选择 kernel。**运行时**加载产物、绑定输入、管理设备、执行 kernel，并报告错误和指标。

![从 eager model 到受监控目标运行时的各个阶段。](assets/dl20-export-runtime.svg){fig-align="center" width="76%" fig-alt="Eager model 依次被捕获、变换、编译并由运行时加载，parity gate 检查代表性输入和形状约束。"}

导出会把假设显式化。依赖任意 Python 状态的分支可能无法捕获；动态维度需要声明范围；不受支持的 custom operator 需要 lowering 或运行时实现。Graph fusion 减少 launch 与中间内存开销，但改变操作顺序也可能改变浮点舍入。因此，每个编译产物都是一种新的数值实现，必须重新通过 parity 与任务质量门。

[PyTorch AOTInductor](https://docs.pytorch.org/docs/main/user_guide/torch_compiler/torch.compiler_aot_inductor.html)使用 `torch.export` 作为 ahead-of-time graph 边界，并可为非 Python 部署打包产物。`torch.compile` 则在保留 Python 工作流的同时优化执行，guard 不满足时可能重新编译。

<details>
<summary><strong>PyTorch：捕获 export graph 并验证编译结果一致性</strong></summary>

```python
export_input = test_x[:16]
baseline_model.eval()
exported_program = torch.export.export(baseline_model, (export_input,))
exported_model = exported_program.module()
with torch.inference_mode():
    eager_output = baseline_model(export_input)
    exported_output = exported_model(export_input)
assert torch.allclose(eager_output, exported_output, atol=1e-6, rtol=1e-5)

# The eager backend exercises Dynamo graph capture without claiming a native-kernel speedup.
compiled_model = torch.compile(copy.deepcopy(baseline_model), backend="eager", fullgraph=True)
with torch.inference_mode():
    compiled_output = compiled_model(export_input)
assert torch.allclose(eager_output, compiled_output, atol=1e-6, rtol=1e-5)
graph_ops = [node.target for node in exported_program.graph.nodes if node.op == "call_function"]
print({"captured_ops": len(graph_ops), "export_parity": True, "compile_parity": True})
```

</details>

生产测试矩阵应覆盖最小、典型与最大形状，空输入或异常输入，数值上困难的取值，以及每个声明的 device/precision profile。除 warm throughput 外，还要追踪 graph break、recompilation count、fallback operator、peak workspace、engine build time 与 cold-start latency。


### **ONNX、TensorRT 与 Triton** {#onnx-tensorrt-triton}

这三个名称处于不同层级：

- **ONNX** 是算子 graph format 与交换契约。**ONNX Runtime** 验证并执行该 graph，把子图分配给硬件 execution provider，并应用 graph optimization。
- **TensorRT** 构建面向 NVIDIA 的优化 inference engine。精度、workspace、tactic selection 与 optimization profile 可能使产物依赖特定硬件和形状。
- **Triton Inference Server** 是服务层，负责 model repository、协议、instance、指标、ensemble 与 scheduler；它可以托管 TensorRT、ONNX Runtime 及其他 backend。

![Triton、ONNX Runtime、TensorRT 与硬件 kernel 的关系。](assets/dl20-runtime-stack.svg){fig-align="center" width="74%" fig-alt="Triton 构成请求服务层，下方是 ONNX Runtime 与 TensorRT 执行路径，两者最终都使用硬件 kernel 和内存。"}

[ONNX Runtime](https://onnxruntime.ai/docs/)强调跨框架与跨语言执行。[TensorRT dynamic-shape 文档](https://docs.nvidia.com/deeplearning/tensorrt/latest/inference-library/work-with-dynamic-shapes.html)要求 optimization profile 为每个运行时维度设置边界。[Triton dynamic batcher](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/batcher.html)合并相容的无状态请求，而 [Model Analyzer](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/model_analyzer/README.html)搜索配置之间的取舍。

<details>
<summary><strong>Python：验证 shape profile 并生成最小 Triton 服务契约</strong></summary>

```python
deployment_manifest = {
    "name": "digits_mlp",
    "version": 1,
    "input": {"name": "pixels", "dtype": "FP32", "shape": [64]},
    "output": {"name": "logits", "dtype": "FP32", "shape": [10]},
    "batch_profile": {"min": 1, "opt": 16, "max": 64},
}


def validate_batch_profile(manifest, observed_batches):
    profile = manifest["batch_profile"]
    if not (1 <= profile["min"] <= profile["opt"] <= profile["max"]):
        raise ValueError("invalid optimization profile ordering")
    unsupported = [batch for batch in observed_batches if not profile["min"] <= batch <= profile["max"]]
    return unsupported


observed = [1, 8, 16, 32, 64]
assert validate_batch_profile(deployment_manifest, observed) == []
triton_config = f'''name: "{deployment_manifest["name"]}"
platform: "onnxruntime_onnx"
max_batch_size: {deployment_manifest["batch_profile"]["max"]}
input [{{ name: "pixels" data_type: TYPE_FP32 dims: [64] }}]
output [{{ name: "logits" data_type: TYPE_FP32 dims: [10] }}]
dynamic_batching {{ preferred_batch_size: [8, 16, 32] max_queue_delay_microseconds: 800 }}'''
assert "dynamic_batching" in triton_config and "dims: [64]" in triton_config
print(triton_config)
```

</details>

导出 graph 并不等于部署完成。需要比较 eager 与运行时输出，确认预期节点被分配给加速器而不是 CPU fallback，检查 engine profile 是否覆盖真实形状，并在接近真实的并发下进行 load test。模型、预处理、运行时、driver 与配置都应一起版本化，因为其中任何一个变化都可能改变行为或性能。


### **云端、边缘与端侧部署** {#cloud-edge-on-device}

云端加速器具有弹性容量、成熟可观测性和大内存，但每个请求都要承担网络和多租户成本。边缘网关把计算移到数据源附近，可以在固定本地预算下汇总多个设备。端侧执行减少网络依赖并可能改善隐私，却受到内存、能耗、温度、二进制大小和算子支持的严格约束。

![云端、边缘与端侧约束共同决定产物选择。](assets/dl20-deployment-targets.svg){fig-align="center" width="72%" fig-alt="云端、边缘与端侧部署框列出各自约束，并共同指向产物选择规则。"}

放置位置是端到端决策。小模型能够节省服务端计算，但如果无线连接启动时间主导延迟，这项收益可能消失。更大的本地模型在持续运行时可能超过热限制。混合设计可以先在本地执行 confidence gate，再把不确定请求升级到云端，但这又带来一致性、隐私与 fallback 问题。除了准确率，还必须考察 package size、peak working memory、cold start、每请求能耗、离线可用性、更新带宽与目标算子覆盖。

<details>
<summary><strong>Python：在明确的部署预算下筛选候选产物</strong></summary>

```python
def parameter_bytes(model):
    return sum(parameter.numel() * parameter.element_size() for parameter in model.parameters())


# Logical compressed sizes are separated from the FP32 teaching tensors used to emulate numerics.
ptq_logical_bytes = weight_count + 4 * (bias_count + 3 + baseline_model.fc1.out_features + baseline_model.fc2.out_features + baseline_model.fc3.out_features)
candidates = [
    {"name": "FP32_teacher", "accuracy": baseline_accuracy, "bytes": parameter_bytes(baseline_model)},
    {"name": "PTQ_W8A8", "accuracy": ptq_accuracy, "bytes": ptq_logical_bytes},
    {"name": "INT4_weight_only", "accuracy": int4_accuracy, "bytes": int4_logical_bytes},
    {"name": "distilled_student", "accuracy": student_accuracy, "bytes": parameter_bytes(student_model)},
]


def feasible_artifacts(items, minimum_accuracy, maximum_kib):
    return [item for item in items if item["accuracy"] >= minimum_accuracy and item["bytes"] / 1024 <= maximum_kib]


edge_choices = feasible_artifacts(candidates, minimum_accuracy=0.90, maximum_kib=40)
assert edge_choices and all(item["bytes"] <= 40 * 1024 for item in edge_choices)
print([{"name": item["name"], "accuracy": round(item["accuracy"], 3), "KiB": round(item["bytes"] / 1024, 1)} for item in edge_choices])
```

</details>

这道初始门只筛选准确率与逻辑产物大小。发布决策还需要目标设备延迟、peak resident memory、能耗、热行为、启动时间以及签名更新/回滚测试。不能用桌面 CPU 计时或逻辑 bit width 推断移动设备速度。


### **监控、漂移、回滚与成本控制** {#monitoring-drift-rollback-cost}

部署改变了模型团队能够获得的证据。输入与系统信号会立即到达，可信标签则可能延迟或完全缺失。因此监控应分为四层：**服务健康**（错误、饱和度、延迟）、**数据健康**（schema、缺失、范围、漂移）、**模型行为**（confidence、abstention、slice metric）以及**业务/安全结果**（最初促使系统建模的延迟后果）。

![从流量、遥测到回滚决策的监控闭环。](assets/dl20-monitoring-loop.svg){fig-align="center" width="72%" fig-alt="流量经过模型进入遥测与延迟质量评估，决策节点可以继续、canary 或回滚。"}

分布漂移是变化的证据，不是质量下降的证明。参考直方图与线上直方图之间的 Jensen-Shannon divergence 有界且对称：

$$JS(P,Q)=\frac{1}{2}D_{KL}(P\|M)+\frac{1}{2}D_{KL}(Q\|M), \qquad M=\frac{P+Q}{2}.$$

较大的 (JS) 只说明特征频率发生变化；只有延迟标签、受控审计或任务专用 proxy 才能确认预测是否退化。还必须分 slice 监控，因为总体稳定可能掩盖某个设备、地区、类别或序列长度区间的失败。

![干净的 Digits 请求以及合成亮度和噪声偏移。](assets/dl20-digits-clean-drift.png){fig-align="center" width="72%" fig-alt="两行图像比较六个干净手写数字与经过亮度偏移和噪声处理的版本，用于演示输入漂移。"}

*数据来源：Alpaydin 与 Kaynak，[UCI Optical Recognition of Handwritten Digits](https://doi.org/10.24432/C50P49)，CC BY 4.0。下方一行是本地生成的亮度/噪声偏移，用于诊断演示。*

<details>
<summary><strong>PyTorch：连接漂移、confidence、质量与回滚策略</strong></summary>

```python
test_y = targets[test_ids]
drift_generator = torch.Generator().manual_seed(2020)
shifted_x = (0.55 * test_x + 0.25 + 0.12 * torch.randn(test_x.shape, generator=drift_generator)).clamp(0, 1)


def prediction_snapshot(model, x, y):
    model.eval()
    with torch.inference_mode():
        probabilities = model(x).softmax(dim=-1)
    return {
        "accuracy": float((probabilities.argmax(1) == y).float().mean()),
        "mean_confidence": float(probabilities.max(dim=1).values.mean()),
    }


def jensen_shannon_histogram(reference, live, bins=20):
    p, _ = np.histogram(reference.numpy(), bins=bins, range=(0, 1), density=False)
    q, _ = np.histogram(live.numpy(), bins=bins, range=(0, 1), density=False)
    p = (p + 1e-8) / (p.sum() + bins * 1e-8)
    q = (q + 1e-8) / (q.sum() + bins * 1e-8)
    m = 0.5 * (p + q)
    return float(0.5 * np.sum(p * np.log(p / m)) + 0.5 * np.sum(q * np.log(q / m)))


clean_snapshot = prediction_snapshot(baseline_model, test_x, test_y)
shifted_snapshot = prediction_snapshot(baseline_model, shifted_x, test_y)
drift_score = jensen_shannon_histogram(features[train_ids].flatten(), shifted_x.flatten())
rollback = shifted_snapshot["accuracy"] < clean_snapshot["accuracy"] - 0.08
assert drift_score > 0 and shifted_snapshot["accuracy"] <= clean_snapshot["accuracy"]
print({"clean": clean_snapshot, "shifted": shifted_snapshot, "JS": round(drift_score, 4), "rollback": rollback})
```

</details>

安全发布应依次使用 shadow traffic、离线重放、小流量 canary 与明确的自动/人工回滚阈值。上线前保留旧产物与预处理 bundle 并真正验证回滚；同时通过 admission limit、输出长度上限、cache quota、autoscaling limit 与每个成功请求成本来限制开销。如果重试、失败或低质量输出增加，只看每个原始 token 的成本是不完整的。


### **准确率-延迟-内存取舍** {#accuracy-latency-memory-tradeoff}

部署优化是多目标问题。如果候选 (A) 的准确率不低于 (B)、延迟不高于 (B)、体积不大于 (B)，并且至少一个维度严格更好，则称 (A) **支配** (B)。不被支配的候选构成 Pareto frontier，产品约束再从中选择一点。加权总分可能掩盖不可接受的硬约束，因此应先执行质量、安全、内存与尾部延迟门，再在可行产物中优化成本。

![部署候选的概念性 Pareto frontier。](assets/dl20-pareto.svg){fig-align="center" width="70%" fig-alt="准确率与延迟/内存成本形成坐标，若干候选位于虚线 Pareto frontier 上，另有一个候选被支配。"}

不同压缩方法移动不同坐标。PTQ 与 weight-only quantization 面向数值表示和带宽；QAT 在选定量化器下恢复质量；剪枝依赖结构化或 sparse-kernel 支持；蒸馏改变架构；batching 提高利用率却增加排队延迟；KV cache 用内存换取避免重复计算。任何方法都不应只依据文件大小或单一 microbenchmark 选择。

<details>
<summary><strong>Python：建立可复现候选表并找出非支配方案</strong></summary>

```python
model_variants = {
    "FP32_teacher": baseline_model,
    "PTQ_W8A8": ptq_model,
    "QAT_fake_quant": qat_model,
    "INT4_simulation": int4_model,
    "pruned_dense_shape": pruned_model,
    "distilled_student": student_model,
}
logical_bytes = {
    "FP32_teacher": parameter_bytes(baseline_model),
    "PTQ_W8A8": ptq_logical_bytes,
    "QAT_fake_quant": ptq_logical_bytes,
    "INT4_simulation": int4_logical_bytes,
    # Unstructured zeros do not shrink a dense artifact without a sparse format.
    "pruned_dense_shape": parameter_bytes(pruned_model),
    "distilled_student": parameter_bytes(student_model),
}

deployment_records = []
for name, model in model_variants.items():
    timing = benchmark_classifier(model, test_x[:128], batch_size=16, repeats=2)
    deployment_records.append({
        "name": name,
        "accuracy": accuracy(model, test_loader),
        "latency_ms": timing["p50_batch_ms"],
        "logical_KiB": logical_bytes[name] / 1024,
    })


def dominates(a, b):
    no_worse = (
        a["accuracy"] >= b["accuracy"]
        and a["latency_ms"] <= b["latency_ms"]
        and a["logical_KiB"] <= b["logical_KiB"]
    )
    strictly_better = (
        a["accuracy"] > b["accuracy"]
        or a["latency_ms"] < b["latency_ms"]
        or a["logical_KiB"] < b["logical_KiB"]
    )
    return no_worse and strictly_better


frontier = [record for record in deployment_records if not any(dominates(other, record) for other in deployment_records if other is not record)]
assert frontier and all(record["accuracy"] > 0.80 for record in deployment_records)
print("all candidates:", [{**r, "accuracy": round(r["accuracy"], 3), "latency_ms": round(r["latency_ms"], 4), "logical_KiB": round(r["logical_KiB"], 1)} for r in deployment_records])
print("local reference frontier:", [record["name"] for record in frontier])
```

</details>

上述低 bit 变体执行的是重构 FP32 或 fake-quantized 操作，因此本地延迟列只是**参考路径诊断**，不是 INT8/INT4 kernel benchmark。必须在每个目标运行时用真实打包产物、真实并发、p95/p99 latency、peak memory、能耗与分 slice 质量重建该表。即使模型 weight 不变，硬件或运行时升级也可能改变 frontier。


### **章节对比与总结** {#chapter-comparison-summary}

高效部署从可信的 eager reference 开始，以可监控、可逆的发布结束。两者之间是一串变换，每一步都会改变假设，因此都需要新的证据。

| 决策 | 主要资源 | 核心收益 | 常见失败 | 必需证据 |
|---|---|---|---|---|
| 动态/连续 batching | scheduler slot | 更高利用率 | 排队与饥饿 | 类真实到达下的延迟分布 |
| KV cache / paging | 加速器内存 | 避免重复前缀投影 | 碎片、驱逐、位置错误 | token 级 parity 与 cache telemetry |
| Speculative decoding | target model call | 每次 target step 接受更多 token | draft 太弱或校正错误 | 分布测试与 TTFT/TPOT |
| PTQ / QAT | 数值表示 | 更小产物与受支持低 bit 计算 | clipping 与敏感层 | 质量、逐层误差、打包 kernel benchmark |
| 仅权重量化 | weight 带宽 | 降低 decode 内存流量 | fallback/反量化开销 | 目标 tokens/s 与 resident memory |
| 剪枝 | 参数结构 | 潜在稀疏或更小 graph | 零值没有带来 kernel 加速 | 真实 sparse/structured runtime 测量 |
| 蒸馏 | 架构大小 | 更小的 dense student | 复制错误或丢失尾部行为 | hard-label baseline 与 slice 评估 |
| 导出 / 编译 | graph 与 kernel | fusion、可移植性、AOT 执行 | graph break、不支持算子、形状未命中 | parity matrix 与 fallback report |
| 监控 / 回滚 | 运行风险 | 发现并限制回退 | 把 proxy drift 误认为质量 | telemetry、延迟标签、已验证回滚 |

<details>
<summary><strong>Python：把最低发布门编码为可执行策略</strong></summary>

```python
def release_gate(candidate, reference_accuracy, maximum_accuracy_drop=0.02, maximum_kib=80):
    checks = {
        "quality": candidate["accuracy"] >= reference_accuracy - maximum_accuracy_drop,
        "memory": candidate["logical_KiB"] <= maximum_kib,
        "latency_measured": candidate["latency_ms"] > 0,
        "rollback_artifact_present": True,
    }
    return checks, all(checks.values())


candidate = next(record for record in deployment_records if record["name"] == "distilled_student")
checks, releasable_in_local_gate = release_gate(candidate, baseline_accuracy, maximum_accuracy_drop=0.04)
assert set(checks) == {"quality", "memory", "latency_measured", "rollback_artifact_present"}
print({"candidate": candidate["name"], "checks": checks, "local_gate": releasable_in_local_gate})
```

</details>

一条实用顺序是：定义请求与质量契约；建立 warm 与 tail baseline；profile 瓶颈；一次只应用一项干预；验证数值和任务 parity；benchmark 目标运行时；在可观测条件下 canary；并始终保留 rollback。优化对象是受约束的完整系统，而不是孤立的 kernel、bit width 或平均值。
